# Abstract screening and human-consensus analysis

Validate inputs, screen abstracts, evaluate saved predictions, and generate figures. These workflows are also available from the command line. Install the package from the repository root with `python -m pip install -e ".[triage]"`.

The reference contains adjudicated full-text labels. Individual annotations are retained for agreement calculations, including ambiguous or missing ratings. Screening sends only the configured bibliographic fields and abstract.

## Inputs and setup

All paths below are relative to the repository root. The included files are ready for local validation. Run the cells in order; leave `RUN_SCREENING = False` for offline analysis.

| Input | Location | For another collection |
| --- | --- | --- |
| Bibliographic metadata, CSV | `data/processed_data/literature_metadata.csv` | Select a CSV or XLSX in `configs/abstract_triage.json`; retain `DOI`, `Article Title`, `Source Title`, `Author Keywords`, `Keywords Plus`, and `Abstract`. |
| Human reference, XLSX | `benchmarks/abstract_triage/ground_truth.xlsx` | Select a one-sheet workbook or CSV with `DOI` and binary `Consensus GT`; retain individual annotation columns for agreement analysis. |
| Screening prompt, TXT | `prompts/abstract_triage.txt` | Selected by `prompt_file` in the configuration. |
| Saved model run, optional | `results/abstract_triage/<run>/` | Set `RUN_DIR` in Section 5 to a run containing its manifest and responses. |

For a four-abstract API walkthrough with an offline request preview, open [the triage demo](../Demo/03_api_demo/api_demo.ipynb). For a small live run, set `max_papers` in the configuration and enable screening. The default human-agreement calculation uses 50,000 bootstrap samples and can take several minutes. Outputs are written under `results/abstract_triage/`.

Implementation: [screening](../src/mofinder/literature/triage.py), [evaluation](../src/mofinder/evaluation/triage.py), and [plotting](../src/mofinder/plotting/triage.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


## 1. Settings and inputs

Edit `configs/abstract_triage.json` to select models, input files, and analysis settings. `benchmark_only: true` selects reference DOIs; `max_papers` can limit a new screening run. Paths are resolved from the configuration's project root.


In [ ]:
from pathlib import Path
import json

from mofinder.display import display_path
from mofinder.config import load_triage_config
from mofinder.literature.triage import validate_inputs, validation_summary, human_agreement, screen
from mofinder.evaluation.triage import load_saved_run, evaluate_run
from mofinder.plotting.triage import plot_results

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/abstract_triage.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MOFinder checkout.")
CONFIG_FILE = ROOT / "configs/abstract_triage.json"
config = load_triage_config(CONFIG_FILE)
validated = validate_inputs(config["input_file"], config["ground_truth_file"],
                            sheet=config["input_sheet"], benchmark_only=config["benchmark_only"],
                            max_papers=config["max_papers"])
print(json.dumps(validation_summary(validated), indent=2))


## 2. Screening prompt

The prompt retains the eligibility rules for cited preparation, separately synthesized MOF precursors, sonication for dissolution, and crystalline-framework evidence. The human annotation notes informed prompt refinement, and the same reference is used for evaluation.


In [ ]:
print(config["prompt_file"].read_text(encoding="utf-8"))


## 3. Screen the abstracts

Change `RUN_SCREENING` to `True` to make model requests. Enter your API key in the hidden prompt when requested, or set `OPENAI_API_KEY` in your environment first. Each completed attempt is saved immediately. Invalid answers and API errors remain unscored.

For a new run, leave `OUTPUT_DIR` as `None` or specify a new folder. To continue an interrupted Python run, supply its existing folder and set `RESUME = True`. Only unrecorded requests are sent; recorded failures are retained. Use a new run to retry recorded failures or change settings.


In [ ]:
RUN_SCREENING = False
OUTPUT_DIR = None
RESUME = False
screening_run = None
if RUN_SCREENING:
    import os
    from getpass import getpass

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ").strip()
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("An API key is required for this run.")

    screening_run = await screen(CONFIG_FILE, output_dir=OUTPUT_DIR, resume=RESUME)
    print("Saved run:", display_path(screening_run.output_dir))
else:
    print("Screening is disabled. Input checks and human-agreement calculations remain available.")


## 4. Human annotation agreement

Agreement uses the individual ratings before consensus. `Y/N` and blank ratings are not forced into a class. Raw pairwise agreement weights observed rater pairs. Fleiss' kappa uses publications with four binary ratings, and nominal Krippendorff's alpha uses available binary ratings. Confidence intervals resample publications.


In [ ]:
agreement = human_agreement(validated["ground_truth"], bootstraps=config["bootstraps"],
                            seed=config["statistics_seed"])
for row in agreement:
    print(f"{row['Statistic']}: {row['Estimate']:.4f} "
          f"(95% CI {row['CI lower']:.4f} to {row['CI upper']:.4f})")


## 5. Evaluate a saved run

Set `RUN_DIR` to a folder containing `run_manifest.json` and `responses.jsonl` or `predictions.csv`. This step makes no model requests. A newly screened run is selected automatically below; otherwise choose the folder explicitly.

`Y` is the positive class. Metrics use valid predictions and retain the full reference denominator in coverage reports. Unanimous and non-unanimous subsets are sensitivity analyses. Paired tests and shared-set figures use common scored publications within each round. Undefined statistics remain undefined. The current reference is compared with the saved reference, and any changes are recorded.

Accuracy, precision, recall, specificity, and NPV use 95% Wilson intervals. F1, balanced accuracy, and MCC use publication-bootstrap percentile intervals. Across-round summaries report means and sample standard deviations (`ddof=1`), without pooling repeated predictions as independent publications. Single-letter Y/N outputs provide no continuous scores for ROC curves, AUC, or probability calibration.


In [ ]:
RUN_DIR = screening_run.output_dir if screening_run is not None else None
# Example: RUN_DIR = ROOT / "results/abstract_triage/your_run"
ANALYSIS_DIR = None  # None creates a new analysis folder within RUN_DIR.
analysis = None
if RUN_DIR is not None:
    analysis = evaluate_run(load_saved_run(
        RUN_DIR, config["ground_truth_file"], output_dir=ANALYSIS_DIR,
        bootstraps=config["bootstraps"], statistics_seed=config["statistics_seed"],
    ))
    print("Analysis folder:", display_path(analysis["output_dir"]))
    for row in analysis["metrics"]:
        print(row)
else:
    print("Set RUN_DIR to analyze saved predictions.")


## 6. Figures and exported results

Generate confusion matrices and performance figures for each configuration and round. Additional figures compare F1 and recall on shared publications and show human vote patterns and consensus labels. Figures are saved as PNG, PDF, and SVG. CSV tables retain the underlying counts and statistical results.


In [ ]:
if analysis is not None:
    figure_paths = plot_results(analysis, show=True)
    print(f"Saved {len(figure_paths)} figure files in {display_path(analysis['output_dir'] / 'figures')}")
